## Использование Gamac на реальных данных
Описание реальных задач приведено в [документации](https://github.com/ITMO-CODE-AI/GaMAC/blob/develop/docs/CASE_RU.md)

### Импортируем нужные библиотеки

In [1]:
import os
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
from PIL import Image

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### 1. Данные промышленных логов (семпл)
Кластеризация логов направлена на автоматическое выделение групп записей, связанных с общими источниками событий, такими как прикладные процессы, системные события или среды исполнения. Основная цель — обнаружить скрытые паттерны в данных, которые позволяют косвенно определить природу возникновения логов, даже если их формат не содержит явных указаний на источник.

Для получения данных нужно выгрузить их из Minio, данные лежат в: datasets/data/synthetic_logs_1kk.csv

И переложить из корня в папку data/

Примечание: выполнение по этому датасету проходит очень долгое время в связи с его объемом

In [2]:
# загрзука данных из MinIO
import os
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [2]:
# Импортируем датафрейм
data = pd.read_csv('../data/synthetic_logs_1kk.csv')

In [3]:
data.head(5)

,object,agent,subject,environment,source,result,result code,tag,log level,label,traceback,protocol,action,property,timestamp
0,missing,missing,thread,missing,24.149.203.18,[<:subject:> <:source:>] File does not exist:<...,missing,missing,error,missing,event interval expired,missing,missing,lifetime 05:00,2025-04-18 10:08:58
1,dfs.FSNamesystem,ocsp.digicert.com:80,out block blk 8855628190418004805,missing,transfer block blk -1132082380757283113 to 10....,<:action:>* NameSystem.addStoredBlock: blockMa...,missing,27052.0,WARN,missing,Redundant addStoredBlock request received for ...,missing,Access denied,power/control problem,2025-03-28 10:08:58
2,spoolsv.exe *64,news.17173.com:80,thread,missing,transfer block blk -1009207079038502874 to 10....,link errors remain current,missing,6708.0,-1,state_change.unavailable,Redundant addStoredBlock request received for ...,missing,missing,power/control problem,2025-04-19 10:08:58
3,chrome.exe,pubads.g.doubleclick.net:443,node-139,missing,84.92.64.33,"<:action:>, <:NUM:> bytes <:*:> <:*:> sent, <:...",missing,missing,missing,missing,timed,missing,close,power/control problem,2025-04-14 10:08:58
4,partition,mini.cpc.sogou.com:80,thread,missing,85.64.10.104,critical,missing,385417.0,-1,status,must start with /,missing,missing,power/control problem,2025-04-16 10:08:58


In [4]:
# По умолчанию кол-во итераций стоит 50
gamac = Gamac(iter_limit=100, target_measures={Internal.BR})

result = gamac.run(table=data)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 1.7002971172332764s, {'BR': -2.3529007114713343} ===
=== MEASURES 0.03184628486633301s, {'BR': -1.9362453504561496} ===
=== ALGO 6.6121416091918945s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 12, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 84.92697024345398s, FailedRun, {'bandwidth': 0.03364232569992478, 'max_iter': 278, 'tol': 8.226330720463991e-05} ===
=== ALGO 24.271040439605713s, FailedRun, {'name': 'DBSCAN', 'eps': 1.3692190941397853, 'eps_sq': 1.8747609277569741, 'min_samples': 6} ===
=== ALGO 7.690703630447388s, FailedRun, {'min_cluster_size': 15, 'min_samples': 10} ===
=== MEASURES 0.0246279239654541s, {'BR': -2.0554529155823054} ===
=== ALGO 51.484442472457886s, SuccessRun, {'threshold': 0.4297222742672191, 'branching_factor': 41, 'n_clusters': 10} ===
=== MEASURES 0.024587392807006836s, {'BR': -2.0469494416801335} ===
=== ALGO 0.3341364860534668s, SuccessRun, {'name': 'KMeans', 'n_clusters': 8, 'max_iter': 100, 'tol': 0.0001, 'ra

In [5]:
# Получение топ-50 меток по лучшему классификатору
print(result.labels[:50])

[12  0 14  4 14  0  0  7  4  9  7 14 12  5  6  9 13 10 10 11  7  5  7  9
  3  0  0  4 12  9  7  5  4  3 10  5 14  0  9 14 10 12 10  3  1  5 10 13
  0  9]


### 2. Данные по анализу цветов RGB-изображений
Кластеризация цветов RGB-изображений направлена на автоматическое группирование пикселей со схожими цветовыми значениями в отдельные кластеры, где каждый кластер представляет собой усреднённый цвет или доминирующий оттенок из соответствующей группы. Основная цель — упростить представление изображения за счёт выделения ключевых цветовых паттернов, что может быть полезно для задач сжатия данных, сегментации объектов, снижения шума или анализа цветовой палитры.

In [6]:
# Импортируем картинки
images = []

# Получаем список файлов в директории
for filename in os.listdir('../data/images/'):
    if 'png' not in filename:
        continue
    file_path = os.path.join('../data/images/', filename)

    # Пытаемся открыть файл как изображение
    with Image.open(file_path) as img:
        images.append(img.copy())

images[:5]

[<PIL.Image.Image image mode=RGBA size=399x382>,
 <PIL.Image.Image image mode=RGBA size=292x256>,
 <PIL.Image.Image image mode=RGBA size=322x262>,
 <PIL.Image.Image image mode=RGBA size=128x142>,
 <PIL.Image.Image image mode=RGBA size=533x775>]

### Запустим подбор

In [7]:
# Для кластеризации передадим константный пустой текст
gamac = Gamac(iter_limit=30)
empty_texts = ['' for i in range(len(images))]

result = gamac.run(image=images, text=empty_texts)

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== Started CVI prediction ===
=== CVI prediction iteration 1/2 ====
=== CVI prediction iteration 2/2 ====
=== Picked SYM in 15.782623052597046s ===
=== MEASURES 0.08414840698242188s, {'SYM': 0.0005034346690834904} ===
=== MEASURES 0.028985261917114258s, {'SYM': 0.03270674730447069} ===
=== ALGO 0.39401721954345703s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 11, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== MEASURES 0.08552384376525879s, {'SYM': 0.0} ===
=== ALGO 1.9595541954040527s, SuccessRun, {'bandwidth': 0.96640807876847, 'max_iter': 222, 'tol': 9.851115540091693e-05} ===
=== MEASURES 0.0794365406036377s, {'SYM': 0.0} ===
=== ALGO 1.1021745204925537s, SuccessRun, {'name': 'DBSCAN', 'eps': 1.3918918906721673, 'eps_sq': 1.9373630353189406, 'min_samples': 9} ===
=== ALGO 0.20662832260131836s, FailedRun, {'min_cluster_size': 7, 'min_samples': 5} ===
=== MEASURES 0.05360102653503418s, {'SYM': 0.00644209907333656} ===
=== ALGO 1.4959404468536377s, SuccessRun, 

In [8]:
# Метки по датасету
print(result.labels)

[ 0  1  0  7  4  4  9  9  9  2  7  9  4  4  1  2  2  0  0  0  0  0 10  6
 10  4  4  6  6  6  7  9  9  9  9  7  7  7  3  9  0  0  0  1  1  1  1  7
  2  7  5  5  6  9  7  7  7  1  1  1  7  7  7  6  7  5  5  5  5  6  5  5
  5  5  5 10  5  5  5  5  5  5  1  0  1  0  1  2  9  2  2  2  9  2  2  9
  7  1  1  7  7  7  8  1  0  0  0  0  9  6  2  2  1  1  1  1  1  4  4  0
  4  4  0  1  4  6  0  1  1  1  1  1  0  0  0  1  1  1  1  1  4  4  1  1
  4  4  0  0  0  1  1  1  1  1  6  0  0  0  7  7  7  7  0  0  0  1  6  6
  4  9  2  4  4  1  3  3  3  3  3  6  7  7  7  1  1  1  0  0  4  4  4  4
  1  0  6  9  6 10  9  9  9  9  3  3  1  1  1  7  7 10 10 10 10  9  9  1
  1  1  2  9  2  2  9  9  7  7 10  9 10  9  4  7  7  8  1  4  4  1  1  1
  1  1  9  2  7  0  1  0  0  7  7  7  4  6  8  8  7  1  1  1  1  1  4  5
  5  5  1  1  7  7  7  4  0  4  4  4  4  0  0  0  0  0  4  9  9  9  7  0
  0  1  0  4  6  4  9  7  7  7  7  1  1  1  1  1  0  0  0  7  7  7  1  0
  0  7  7  7  7  4  4  7  3  3  3  3  3  3  3  3  3